# Phân cụm K - mean
5 cụm chủ đề chính và các từ khoá đại diện cho mỗi cụm


In [2]:
import pandas as pd
import joblib
from scipy import sparse
import sys

sys.path.append('../')
from src.mining.clustering import TopicClustering

df = pd.read_csv('../data/processed/cleaned_reviews.csv')
tfidf_matrix = sparse.load_npz('../data/processed/tfidf_matrix.npz')
vectorizer = joblib.load('../outputs/models/tfidf_vectorizer.pkl')

clusterer = TopicClustering(n_clusters=5, random_state=42)
df['cluster'] = clusterer.fit_predict(tfidf_matrix)

top_terms = clusterer.get_top_terms_per_cluster(vectorizer, n_terms=5)
for cluster, terms in top_terms.items():
    print(f"{cluster}: {', '.join(terms)}")

display(df[['reviews.rating', 'cleaned_text', 'cluster']].head())

/var/folders/s3/h6hv6qkd01d4qr6_nb2f3m3m0000gn/T/ipykernel_9170/2948834275.py:9: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/processed/cleaned_reviews.csv')


Cluster 0: great, hotel, stay, place, location
Cluster 1: enjoyed, enjoyed stay, stay, really enjoyed, really
Cluster 2: lobby, window open, complementary, dry, adjacent
Cluster 3: clean, staff, nice, friendly, room
Cluster 4: room, hotel, night, bed, desk


,reviews.rating,cleaned_text,cluster
0,4.0,pleasant min walk along sea front water bus re...,4
1,5.0,really lovely hotel stayed top floor surprised...,3
2,5.0,ett mycket bra hotell det som drog ner betyget...,0
3,5.0,stayed four night october hotel staff welcomin...,3
4,5.0,loved staying island lido need take water veni...,0


# thực thi Khai phá Luật kết hợp (Apriori):

In [3]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

df['Sentiment'] = df['reviews.rating'].apply(lambda x: 'Positive' if x >= 4 else 'Negative')

aspects = {
    'Room': ['room', 'bed', 'bathroom', 'shower', 'clean'],
    'Service': ['staff', 'service', 'reception', 'checkin', 'friendly'],
    'Food': ['breakfast', 'food', 'restaurant', 'coffee'],
    'Location': ['location', 'walk', 'close', 'near', 'beach']
}

encoded_data = pd.DataFrame()

for aspect, keywords in aspects.items():
    encoded_data[f'Aspect_{aspect}'] = df['cleaned_text'].apply(
        lambda x: 1 if any(word in str(x).split() for word in keywords) else 0
    )

encoded_data['Sentiment_Positive'] = (df['Sentiment'] == 'Positive').astype(int)
encoded_data['Sentiment_Negative'] = (df['Sentiment'] == 'Negative').astype(int)

frequent_itemsets = apriori(encoded_data, min_support=0.05, use_colnames=True)

rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)

top_negative_rules = rules[
    rules['consequents'].apply(lambda x: 'Sentiment_Negative' in str(x))
].sort_values(by='lift', ascending=False)

top_positive_rules = rules[
    rules['consequents'].apply(lambda x: 'Sentiment_Positive' in str(x))
].sort_values(by='lift', ascending=False)

display(top_negative_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(5))
display(top_positive_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(5))

/Users/vominhquan/miniconda3/lib/python3.13/site-packages/mlxtend/frequent_patterns/fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


,antecedents,consequents,support,confidence,lift
39,(Aspect_Room),"(Sentiment_Negative, Aspect_Service)",0.070003,0.116556,1.259749
53,(Aspect_Room),"(Sentiment_Negative, Aspect_Food)",0.062288,0.103710,1.259603
7,(Aspect_Room),(Sentiment_Negative),0.228478,0.380419,1.115243


,antecedents,consequents,support,confidence,lift
122,(Aspect_Food),"(Aspect_Room, Sentiment_Positive, Aspect_Locat...",0.053351,0.171060,1.643529
120,"(Aspect_Room, Aspect_Location)","(Aspect_Food, Sentiment_Positive)",0.053351,0.360000,1.568284
92,"(Aspect_Food, Aspect_Service)","(Aspect_Room, Sentiment_Positive)",0.089485,0.577137,1.550953
95,(Aspect_Food),"(Aspect_Room, Sentiment_Positive, Aspect_Service)",0.089485,0.286915,1.539108
125,(Aspect_Location),"(Aspect_Room, Aspect_Food, Sentiment_Positive)",0.053351,0.240823,1.523064
